# 03 Modeling

This notebook trains and evaluates predictive models using the processed dataset. It is important to focus on the time-aware evaluation and training, for this reason we will apply a chronological split with a 24-h gap to avoid leakage from rolling windows.

## 3.1 Load processed data

In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

In [3]:
PROCESSED_DIR = Path("../data/processed")
DATASET_PATH = PROCESSED_DIR / "dataset.csv"

In [4]:
dataset = pd.read_csv(DATASET_PATH, parse_dates=["datetime"])
dataset.head(3)

,machineID,datetime,volt,rotate,pressure,vibration,volt_mean_24h,volt_std_24h,rotate_mean_24h,rotate_std_24h,...,error4_count_24h,error5_count_24h,comp1,comp2,comp3,comp4,model,age,failure_binary,failure_comp
0,1,2015-01-01 06:00:00,176.217853,418.504078,113.077935,45.087686,176.217853,NaN,418.504078,NaN,...,0.0,0.0,NaN,NaN,NaN,NaN,model3,18,0,none
1,1,2015-01-01 07:00:00,162.879223,402.747490,95.460525,43.413973,169.548538,9.431836,410.625784,11.141591,...,0.0,0.0,NaN,NaN,NaN,NaN,model3,18,0,none
2,1,2015-01-01 08:00:00,170.989902,527.349825,75.237905,34.178847,170.028993,6.721032,449.533798,67.849599,...,0.0,0.0,NaN,NaN,NaN,NaN,model3,18,0,none


## 3.2 Target and Feature Data

In [40]:
TARGET_MULTICLASS = "failure_comp"      
TARGET_BINARY     = "failure_binary"    

# Columns to drop from X
drop_cols = {"machineID", "datetime", TARGET_MULTICLASS, TARGET_BINARY}

# One-hot 'model' if exists
categoricals = []
if "model" in dataset.columns:
    categoricals.append("model")

# Build X, y 
X = pd.get_dummies(
    dataset.drop(columns=[c for c in drop_cols if c in dataset.columns]),
    columns=categoricals,
    drop_first=False
)

bool_cols = X.columns[X.dtypes == "bool"]
if len(bool_cols):
    X[bool_cols] = X[bool_cols].astype(np.int8)

In [41]:
y_mc = dataset[TARGET_MULTICLASS].astype("category")

y_bin = dataset[TARGET_BINARY].astype(int)

In [42]:
print(X.shape, y_mc.shape, y_bin.shape)
X.head(3)

(876100, 31) (876100,) (876100,)


,volt,rotate,pressure,vibration,volt_mean_24h,volt_std_24h,rotate_mean_24h,rotate_std_24h,pressure_mean_24h,pressure_std_24h,...,error5_count_24h,comp1,comp2,comp3,comp4,age,model_model1,model_model2,model_model3,model_model4
0,176.217853,418.504078,113.077935,45.087686,176.217853,NaN,418.504078,NaN,113.077935,NaN,...,0.0,NaN,NaN,NaN,NaN,18,0,0,1,0
1,162.879223,402.747490,95.460525,43.413973,169.548538,9.431836,410.625784,11.141591,104.269230,12.457390,...,0.0,NaN,NaN,NaN,NaN,18,0,0,1,0
2,170.989902,527.349825,75.237905,34.178847,170.028993,6.721032,449.533798,67.849599,94.592122,18.934956,...,0.0,NaN,NaN,NaN,NaN,18,0,0,1,0


## 3.3 Time-aware split with gap

We split by date threshold: train on earlier dates, test on later dates and insert a 24-hour gap to avoid overlaping on rolling windows.

In [43]:
TRAIN_FRACTION = 0.80
GAP_HOURS = 24

In [44]:
all_times = (
    dataset["datetime"]
    .dropna()
    .sort_values()
    .unique()
)

In [45]:
cut_idx = int(np.floor(TRAIN_FRACTION * len(all_times)))
cut_idx = min(max(cut_idx, 1), len(all_times) - 2)
cut_time = pd.to_datetime(all_times[cut_idx])
cut_time

Timestamp('2015-10-20 06:00:00')

In [46]:
gap = pd.Timedelta(hours=GAP_HOURS)
last_train  = cut_time - gap     
first_test  = cut_time

In [47]:
print("Split summary:")
print(f"Train ends before: {last_train}  ")
print(f"Gap: [{last_train}, {first_test}] to be discarded")
print(f"Test starts after: {first_test}    ")

Split summary:
Train ends before: 2015-10-19 06:00:00  
Gap: [2015-10-19 06:00:00, 2015-10-20 06:00:00] to be discarded
Test starts after: 2015-10-20 06:00:00    


In [48]:
mask_train = dataset["datetime"] < last_train
mask_test  = dataset["datetime"] > first_test

X_train, y_train_mc = X[mask_train], y_mc[mask_train]
X_test,  y_test_mc  = X[mask_test],  y_mc[mask_test]

In [49]:
print("Shapes:")
print("X_train:", X_train.shape, ", y_train_mc:", y_train_mc.shape)
print("X_test: ", X_test.shape,  ", y_test_mc: ", y_test_mc.shape)

Shapes:
X_train: (698400, 31) , y_train_mc: (698400,)
X_test:  (175200, 31) , y_test_mc:  (175200,)


In [74]:
y_train_labels = y_train_mc.to_numpy() if hasattr(y_train_mc, "to_numpy") else y_train_mc
y_test_labels  = y_test_mc.to_numpy()  if hasattr(y_test_mc,  "to_numpy")  else y_test_mc

all_classes = sorted(pd.Index(pd.unique(y_train_labels)).tolist())
label2id = {c: i for i, c in enumerate(all_classes)}
id2label = {i: c for c, i in label2id.items()}

y_train_enc = np.array([label2id[c] for c in y_train_labels], dtype=int)
y_test_enc  = np.array([label2id[c] for c in y_test_labels],  dtype=int)
num_classes = len(all_classes)
print("Classes:", all_classes)

Classes: ['comp1', 'comp2', 'comp3', 'comp4', 'none']


In [77]:
X_train_np = X_train.values
X_test_np  = X_test.values

## 3.4 Modeling

On the following section we will make a benchmarking between three models for benchmarking:
* HistGradientBoostingClassifier
* XGBoost
* LightGBM

In [66]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix, make_scorer
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

from sklearn.metrics import f1_score, classification_report, confusion_matrix, make_scorer
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier

In [51]:
rng = np.random.RandomState(42)

The following section creates a scoring function that computes the macro-average F1 score:
* It computes F1 separately for each class.
* Then it takes the unweighted average across all classes. 
Additionally, we have a `failure-only macro F1`. This ignores the `none`class and evaluates the model only on the actual failures. 

In [57]:
macro_f1 = make_scorer(f1_score, average="macro")

def failure_only_macro_f1(y_true, y_pred):
    y_true = pd.Series(y_true)
    y_pred = pd.Series(y_pred)
    mask = y_true != "none"
    if mask.sum() == 0:
        return 0.0  # no failures in fold: return neutral score
    return f1_score(y_true[mask], y_pred[mask], average="macro")

fail_macro_f1 = make_scorer(failure_only_macro_f1)
PRIMARY_SCORER = fail_macro_f1

In [58]:
tscv = TimeSeriesSplit(n_splits=3)

### 3.4.1 HistGradientBoosting

In [ ]:
hgb = HistGradientBoostingClassifier(
    loss="log_loss",
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42
)

param_hgb = {
    "max_iter":      [200, 400],
    "learning_rate": [0.05, 0.1],
    "max_depth":     [None, 6],
}

grid_hgb = GridSearchCV(
    estimator=hgb,
    param_grid=param_hgb,
    scoring="f1_macro",    
    cv=3,                   
    n_jobs=-1,
    verbose=1
)

In [80]:
grid_hgb.fit(X_train_np, y_train_enc) 
best_hgb = grid_hgb.best_estimator_

Fitting 3 folds for each of 8 candidates, totalling 24 fits


In [ ]:
print("HGB best params:", grid_hgb.best_params_, ", CV f1_macro:", grid_hgb.best_score_)

HGB best params: {'learning_rate': 0.05, 'max_depth': None, 'max_iter': 200}  | CV f1_macro: 0.9790731033907027


In [84]:
y_pred_hgb = best_hgb.predict(X_test_np)
print("Macro F1:", f1_score(y_test_enc, y_pred_hgb, average="macro"))
print(classification_report(y_test_enc, y_pred_hgb, target_names=all_classes, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test_enc, y_pred_hgb))

Macro F1: 0.9912919738223536
              precision    recall  f1-score   support

       comp1     0.9634    0.9828    0.9730       696
       comp2     1.0000    1.0000    1.0000      1149
       comp3     0.9759    0.9981    0.9869       528
       comp4     0.9935    1.0000    0.9968       768
        none     0.9999    0.9998    0.9998    172059

    accuracy                         0.9997    175200
   macro avg     0.9866    0.9961    0.9913    175200
weighted avg     0.9997    0.9997    0.9997    175200

Confusion matrix:
 [[   684      0      0      0     12]
 [     0   1149      0      0      0]
 [     1      0    527      0      0]
 [     0      0      0    768      0]
 [    25      0     13      5 172016]]


### 3.4.XGBClassifier

In [85]:
from xgboost import XGBClassifier

In [86]:
xgb = XGBClassifier(
    objective="multi:softmax",   
    num_class=num_classes,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

param_xgb = {
    "n_estimators":  [200, 400],
    "max_depth":     [4, 6],
    "learning_rate": [0.05, 0.1],
}

grid_xgb = GridSearchCV(
    estimator=xgb,
    param_grid=param_xgb,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=1
)

In [87]:
grid_xgb.fit(X_train_np, y_train_enc)
best_xgb = grid_xgb.best_estimator_

Fitting 3 folds for each of 8 candidates, totalling 24 fits


In [88]:
print("XGB best params:", grid_xgb.best_params_, ",  CV f1_macro:", grid_xgb.best_score_)

XGB best params: {'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 400} ,  CV f1_macro: 0.9827498972525173


In [89]:
y_pred_xgb = best_xgb.predict(X_test_np)
print("Macro F1:", f1_score(y_test_enc, y_pred_xgb, average="macro"))
print(classification_report(y_test_enc, y_pred_xgb, target_names=all_classes, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test_enc, y_pred_xgb))

Macro F1: 0.9953432863296335
              precision    recall  f1-score   support

       comp1     0.9718    0.9885    0.9801       696
       comp2     1.0000    1.0000    1.0000      1149
       comp3     1.0000    1.0000    1.0000       528
       comp4     0.9935    1.0000    0.9968       768
        none     1.0000    0.9999    0.9999    172059

    accuracy                         0.9998    175200
   macro avg     0.9930    0.9977    0.9953    175200
weighted avg     0.9998    0.9998    0.9998    175200

Confusion matrix:
 [[   688      0      0      0      8]
 [     0   1149      0      0      0]
 [     0      0    528      0      0]
 [     0      0      0    768      0]
 [    20      0      0      5 172034]]
